# 00 — Data Pipeline: Investment Thesis

Dieses Notebook lädt historische Kursdaten für beide Thesen:
- **Hydrogen Long:** NEL.OL, AI.PA (Air Liquide), LIN (Linde)
- **AI Valuation Short:** NVDA, MSFT, META, PLTR, AI (C3.ai)

Daten werden als Parquet-Dateien gespeichert 

In [1]:
import yfinance as yf
import polars as pl
from pathlib import Path
import pyarrow.parquet as pq

In [2]:
DATA_DIR = Path("../data/thesis")
DATA_DIR.mkdir(exist_ok=True)

In [3]:
TICKERS = {
    "hydrogen": ["NEL.OL", "AI.PA", "LIN"],
    "ai_short": ["NVDA", "MSFT", "META", "PLTR", "AI"], 
}

In [4]:
all_tickers = [ticker for sublist in TICKERS.values() for ticker in sublist]
all_tickers

['NEL.OL', 'AI.PA', 'LIN', 'NVDA', 'MSFT', 'META', 'PLTR', 'AI']

In [5]:
data = yf.download(all_tickers, start="2020-01-01", end="2026-02-27") 

[*********************100%***********************]  8 of 8 completed


In [6]:
print(type(data))


<class 'pandas.DataFrame'>


In [7]:
#data = pl.from_pandas(data)

In [8]:
data 

Price       Close                                                            \
Ticker         AI       AI.PA         LIN        META        MSFT    NEL.OL   
Date                                                                          
2020-01-02    NaN   93.452423  193.385406  208.324783  152.158371  8.910163   
2020-01-03    NaN   93.157387  188.356705  207.222504  150.263748  8.650449   
2020-01-06    NaN   92.530434  187.558334  211.125229  150.652176  8.385742   
2020-01-07    NaN   92.124763  187.962082  211.582016  149.278534  8.505609   
2020-01-08    NaN   92.198524  190.311356  213.727051  151.656281  8.525587   
...           ...         ...         ...         ...         ...       ...   
2026-02-20  10.38  175.559998  496.510010  655.659973  397.230011  2.104000   
2026-02-23   9.79  174.759995  498.190002  637.250000  384.470001  2.034000   
2026-02-24  10.12  177.300003  504.000000  639.299988  389.000000  2.030000   
2026-02-25  10.31  178.300003  508.269989  653.690002  400.600006  2.024000   
2026-02-26   8.40  179.500000  498.510010  657.010010  401.720001  2.052000   

Price                                High              ...        Open  \
Ticker            NVDA        PLTR     AI       AI.PA  ...        NVDA   
Date                                                   ...               
2020-01-02    5.971078         NaN    NaN   93.894973  ...    5.942207   
2020-01-03    5.875504         NaN    NaN   93.415540  ...    5.851362   
2020-01-06    5.900144         NaN    NaN   92.788587  ...    5.782171   
2020-01-07    5.971576         NaN    NaN   93.268031  ...    5.928518   
2020-01-08    5.982775         NaN    NaN   92.382922  ...    5.967344   
...                ...         ...    ...         ...  ...         ...   
2026-02-20  189.820007  135.240005  10.92  175.559998  ...  186.570007   
2026-02-23  191.550003  130.600006  10.30  176.320007  ...  191.399994   
2026-02-24  192.850006  128.839996  10.18  177.699997  ...  191.490005   
2026-02-25  195.559998  134.190002  10.53  179.020004  ...  194.449997   
2026-02-26  184.889999  135.940002   8.71  179.820007  ...  194.270004   

Price                       Volume                                    \
Ticker            PLTR          AI      AI.PA        LIN        META   
Date                                                                   
2020-01-02         NaN         NaN   860273.0  2604100.0  12077100.0   
2020-01-03         NaN         NaN   861057.0  2899300.0  11188400.0   
2020-01-06         NaN         NaN   889140.0  2395300.0  17058900.0   
2020-01-07         NaN         NaN  1047002.0  2589200.0  14912400.0   
2020-01-08         NaN         NaN  1108034.0  1327300.0  13475000.0   
...                ...         ...        ...        ...         ...   
2026-02-20  132.365005   7418000.0  1499540.0  3202400.0  14183500.0   
2026-02-23  132.039993   7058200.0   937015.0  2159200.0   8605900.0   
2026-02-24  129.005005   5547900.0   838687.0  2761700.0  10135600.0   
2026-02-25  130.610001  10067400.0   930642.0  3483400.0  11330700.0   
2026-02-26  133.845001  30936500.0   738791.0  1945600.0  10637700.0   

Price                                                        
Ticker            MSFT      NEL.OL         NVDA        PLTR  
Date                                                         
2020-01-02  22622100.0  11638331.0  237536000.0         NaN  
2020-01-03  21116200.0  14095589.0  205384000.0         NaN  
2020-01-06  20813700.0  14637804.0  262636000.0         NaN  
2020-01-07  21634100.0   9890112.0  314856000.0         NaN  
2020-01-08  27746500.0   8835941.0  277108000.0         NaN  
...                ...         ...          ...         ...  
2026-02-20  34015200.0   1739502.0  178422300.0  53726800.0  
2026-02-23  43238300.0   4126335.0  171584800.0  52512200.0  
2026-02-24  33884700.0   5502210.0  175123600.0  47121400.0  
2026-02-25  43625500.0   2433577.0  250637100.0  53078300.0  
2026-02-26  34405900.0   7044780.0  36080

In [9]:
collect_prices = (
    pl.from_pandas(
        data["Close"]
        .reset_index()          # Date vom Index → normale Spalte
    )
    .unpivot(
        index="Date",
        variable_name="ticker",
        value_name="close"
    )
    .rename({"Date": "date"})
    .filter(pl.col("date") >= pl.lit("2020-01-01").str.to_date())
    .drop_nulls()               # Zeilen ohne Kurs raus (z.B. NEL vor IPO)
)

collect_prices.write_parquet(DATA_DIR / "prices.parquet")
collect_prices.head(8)


date,ticker,close
datetime[ms],str,f64
2020-12-09 00:00:00,"""AI""",92.489998
2020-12-10 00:00:00,"""AI""",130.0
2020-12-11 00:00:00,"""AI""",119.580002
2020-12-14 00:00:00,"""AI""",102.360001
2020-12-15 00:00:00,"""AI""",102.0
2020-12-16 00:00:00,"""AI""",113.690002
2020-12-17 00:00:00,"""AI""",117.239998
2020-12-18 00:00:00,"""AI""",137.589996


In [10]:
nvda = yf.Ticker("NVDA")
print(nvda.info.keys())


dict_keys(['address1', 'city', 'state', 'zip', 'country', 'phone', 'website', 'industry', 'industryKey', 'industryDisp', 'sector', 'sectorKey', 'sectorDisp', 'longBusinessSummary', 'fullTimeEmployees', 'companyOfficers', 'auditRisk', 'boardRisk', 'compensationRisk', 'shareHolderRightsRisk', 'overallRisk', 'governanceEpochDate', 'compensationAsOfEpochDate', 'irWebsite', 'executiveTeam', 'maxAge', 'priceHint', 'previousClose', 'open', 'dayLow', 'dayHigh', 'regularMarketPreviousClose', 'regularMarketOpen', 'regularMarketDayLow', 'regularMarketDayHigh', 'dividendRate', 'dividendYield', 'exDividendDate', 'payoutRatio', 'fiveYearAvgDividendYield', 'beta', 'trailingPE', 'forwardPE', 'volume', 'regularMarketVolume', 'averageVolume', 'averageVolume10days', 'averageDailyVolume10Day', 'bid', 'ask', 'bidSize', 'askSize', 'marketCap', 'nonDilutedMarketCap', 'fiftyTwoWeekLow', 'fiftyTwoWeekHigh', 'allTimeHigh', 'allTimeLow', 'priceToSalesTrailing12Months', 'fiftyDayAverage', 'twoHundredDayAverage', 

In [11]:
[k for k in nvda.info.keys() if "cash" in k.lower()]


['totalCash', 'totalCashPerShare', 'freeCashflow', 'operatingCashflow']

In [12]:

[k for k in nvda.info.keys() if "book" in k.lower()]


['bookValue', 'priceToBook']

In [14]:
def collect_fundamentals(tickers: list) -> pl.DataFrame:
    rows = []
    for ticker in tickers:
        info = yf.Ticker(ticker).info
        rows.append({
            "ticker":          ticker,
            "market_cap":      info.get("marketCap"),
            "pe_trailing":     info.get("trailingPE"),
            "pe_forward":      info.get("forwardPE"),
            "ev_ebitda":       info.get("enterpriseToEbitda"),
            "price_to_book":   info.get("priceToBook"),
            "revenue_growth":  info.get("revenueGrowth"),    # YoY, als Dezimal (0.12 = 12%)
            "free_cashflow":   info.get("freeCashflow"),
            "gross_margin":    info.get("grossMargins"),
            "operating_margin":info.get("operatingMargins"),
        })
    return pl.DataFrame(rows)


In [15]:
fundamentals = collect_fundamentals(all_tickers)
fundamentals.write_parquet(DATA_DIR / "fundamentals.parquet")
fundamentals


ticker,market_cap,pe_trailing,pe_forward,ev_ebitda,price_to_book,revenue_growth,free_cashflow,gross_margin,operating_margin
str,i64,f64,f64,f64,f64,f64,i64,f64,f64
"""NEL.OL""",3723868672,null,-9.528737,-8.71,0.953861,-0.132,-154409744,0.63586,-2.54977
"""AI.PA""",101532942336,28.8,22.266554,14.334,3.866963,-0.034,1555575040,0.64175,0.19473
"""LIN""",236242960384,34.84689,26.166513,19.482,6.1809382,0.058,4786500096,0.48835,0.28172
"""NVDA""",4441738575872,37.21996,17.09596,31.934,28.23702,0.732,58128998400,0.71068,0.65024
"""MSFT""",2963958136832,24.95557,21.164682,16.833,7.579397,0.167,53640626176,0.68586,0.47094
"""META""",1639607173120,27.629156,18.065216,16.126,7.548651,0.238,23432374272,0.81999,0.41315
"""PLTR""",347789099008,230.81985,78.68305,223.077,47.075592,0.7,1260918400,0.82367,0.40901
"""AI""",1218881280,null,-10.206855,-1.239,1.7087017,-0.461,16967376,0.43481,-2.63627


In [16]:
def collect_analysts(tickers: list) -> pl.DataFrame:
    rows = []
    for ticker in tickers:
        info = yf.Ticker(ticker).info
        rows.append({
             "ticker":         ticker,    
           "recommendation":     info.get("recommendationKey"),   # "buy", "hold", "sell"
"target_mean":        info.get("targetMeanPrice"),
"target_high":        info.get("targetHighPrice"),
"target_low":         info.get("targetLowPrice"),
"num_analysts":       info.get("numberOfAnalystOpinions"),
"short_interest":     info.get("shortPercentOfFloat"),

        })
    return pl.DataFrame(rows)


In [17]:
analysts = collect_analysts(all_tickers)

In [19]:
analysts = collect_analysts(all_tickers)
analysts.write_parquet(DATA_DIR / "analysts.parquet")
analysts

ticker,recommendation,target_mean,target_high,target_low,num_analysts,short_interest
str,str,f64,f64,f64,i64,f64
"""NEL.OL""","""underperform""",2.18333,4.2,1.0,12,null
"""AI.PA""","""buy""",195.5238,216.0,154.0,21,null
"""LIN""","""buy""",512.4308,565.0,381.0,26,0.0131
"""NVDA""","""strong_buy""",263.81775,380.0,140.0,58,0.0109
"""MSFT""","""strong_buy""",595.99567,730.0,392.0,53,0.0076
"""META""","""strong_buy""",863.19934,1144.0,700.0,59,0.0124
"""PLTR""","""buy""",184.48808,260.0,70.0,26,0.0235
"""AI""","""hold""",9.45455,17.0,6.0,11,0.3174
